### I need to get 3D descriptors for df_TgDHFR

In [1]:
import pandas as pd
from padelpy import from_smiles
from padelpy import padeldescriptor

In [2]:
df = pd.read_csv('./1-Training_Database/V2-combined_bd_chmbl_unique.csv')
df.head(2)

,cid,Smiles,Standard Value,pIC50 Value
0,23647975.0,COc1cc(cc(Cc2cnc(N)nc2N)c1OC)C#CCCCC(O)=O,34.0,7.468521
1,5578.0,COc1cc(Cc2cnc(N)nc2N)cc(OC)c1OC,2722.0,5.565112


In [3]:
df.shape

(875, 4)

In [6]:
unique_smiles = df['Smiles'].unique()

len(unique_smiles)

870

# 1. GO of structures using SMILES:

In [7]:
from rdkit import Chem
from rdkit.Chem import AllChem

smiles_list = unique_smiles  # Example SMILES
num_conformers = 10  # Number of conformers to generate

# Open the SDF writer
writer = Chem.SDWriter('combined_molecules_test_new_mmff94s.sdf')

# Set ETKDGv3 parameters for conformer generation
etkdg_params = AllChem.ETKDGv3()
etkdg_params.numThreads = 0  # Use all available CPU threads

for smiles in smiles_list:
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:  # Check if molecule conversion is successful
            print(f"Invalid SMILES: {smiles}")
            continue
        
        mol = Chem.AddHs(mol)  # Add hydrogens
        
        # Generate multiple conformers with ETKDGv3
        ids = AllChem.EmbedMultipleConfs(mol, numConfs=num_conformers, params=etkdg_params)
        
        if len(ids) == 0:  # Check if conformer generation is successful
            print(f"Conformer generation failed for molecule: {smiles}")
            continue

        # Optimize each conformer using MMFF94s and get the energy
        energies = []
        for conf_id in ids:
            try:
                # Optimize the molecule using MMFF94s force field
                mmff_props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant="MMFF94s")
                ff = AllChem.MMFFGetMoleculeForceField(mol, mmff_props, confId=conf_id)
                ff.Minimize()
                
                # Get the energy of the conformer
                energy = ff.CalcEnergy()
                energies.append(energy)
            except Exception as e:
                print(f"Optimization failed for conformer {conf_id} of {smiles}: {e}")
                energies.append(float('inf'))  # Assign a high energy if optimization fails
        
        if all(e == float('inf') for e in energies):  # Check if all optimizations failed
            print(f"All conformer optimizations failed for molecule: {smiles}")
            continue
        
        # Get the conformer with the lowest energy
        min_energy_idx = energies.index(min(energies))
        
        # Create a new molecule with the lowest energy conformer
        lowest_energy_mol = Chem.Mol(mol)
        lowest_energy_mol.RemoveAllConformers()
        lowest_energy_mol.AddConformer(mol.GetConformer(min_energy_idx), assignId=True)
        lowest_energy_mol.SetProp('_Name', smiles)  # Set the name to the SMILES string

        # Write the molecule to the SDF file
        writer.write(lowest_energy_mol)

    except Exception as e:
        print(f"An error occurred for molecule {smiles}: {e}")

# Close the SDF writer
writer.close()

Optimization failed for conformer 0 of CCc1nc(N)nc(N)c1-c1ccc(Cl)c([N+]#N)c1.F[B-](F)(F)F: 'NoneType' object has no attribute 'Minimize'
Optimization failed for conformer 1 of CCc1nc(N)nc(N)c1-c1ccc(Cl)c([N+]#N)c1.F[B-](F)(F)F: 'NoneType' object has no attribute 'Minimize'
Optimization failed for conformer 2 of CCc1nc(N)nc(N)c1-c1ccc(Cl)c([N+]#N)c1.F[B-](F)(F)F: 'NoneType' object has no attribute 'Minimize'
Optimization failed for conformer 3 of CCc1nc(N)nc(N)c1-c1ccc(Cl)c([N+]#N)c1.F[B-](F)(F)F: 'NoneType' object has no attribute 'Minimize'
Optimization failed for conformer 4 of CCc1nc(N)nc(N)c1-c1ccc(Cl)c([N+]#N)c1.F[B-](F)(F)F: 'NoneType' object has no attribute 'Minimize'
Optimization failed for conformer 5 of CCc1nc(N)nc(N)c1-c1ccc(Cl)c([N+]#N)c1.F[B-](F)(F)F: 'NoneType' object has no attribute 'Minimize'
Optimization failed for conformer 6 of CCc1nc(N)nc(N)c1-c1ccc(Cl)c([N+]#N)c1.F[B-](F)(F)F: 'NoneType' object has no attribute 'Minimize'
Optimization failed for conformer 7 of CC

# 2. 3D calculation using PADEL

In [8]:
%%time
from padelpy import padeldescriptor

# Calculate 3D descriptors using padeldescriptor
padeldescriptor(
    mol_dir='combined_molecules_test_new_mmff94s.sdf',
    d_file='V2-TgDHFR-3d-descriptors_ligand_GO_mmff94s.csv',
    fingerprints=False,
    d_3d=True,
)

# Read the results from the CSV file and create the DataFrame
df_ligands_3d = pd.read_csv('V2-TgDHFR-3d-descriptors_ligand_GO_mmff94s.csv')

# Ensure the first column is SMILES
df_ligands_3d.rename(columns={'Name': 'canonicalsmiles'}, inplace=True)

# Display the DataFrame
df_ligands_3d

CPU times: user 52.3 ms, sys: 6.01 ms, total: 58.3 ms
Wall time: 9.1 s


,canonicalsmiles,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,TDB8u,TDB9u,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,COc1cc(cc(Cc2cnc(N)nc2N)c1OC)C#CCCCC(O)=O,1.254996,2.201673,3.103689,3.962396,4.719510,5.557423,6.310768,7.104393,7.897998,...,0.671270,0.270792,0.568703,0.404621,0.409698,22.863896,123.556813,272.298100,0.506905,1.383022
1,COc1cc(Cc2cnc(N)nc2N)cc(OC)c1OC,1.250840,2.176286,3.121375,3.917018,4.606636,5.472777,6.070544,6.997690,7.448239,...,0.681195,0.223644,0.526743,0.420964,0.415681,15.650156,58.403017,129.623380,0.521793,1.363389
2,COc1ccc(OC)c(Cc2cnc3nc(N)nc(N)c3c2C)c1,1.260573,2.205699,3.164785,4.008417,4.709748,5.487354,6.298379,6.942693,7.685606,...,0.684686,0.233761,0.462430,0.432066,0.419386,17.602439,72.799899,161.593140,0.527029,1.313881
3,COc1ccc(OCCCC(O)=O)cc1Cc1cnc2nc(N)nc(N)c2c1C,1.261814,2.211434,3.104852,3.977466,4.734474,5.527540,6.341378,7.124420,7.820012,...,0.674622,0.267538,0.485090,0.455930,0.434351,22.911835,123.353904,271.825209,0.511934,1.375371
4,COc1ccc(OCCCCC(O)=O)cc1Cc1cnc2nc(N)nc(N)c2c1C,1.260856,2.208210,3.088044,3.951555,4.730086,5.517038,6.389618,7.196127,8.021782,...,0.805610,0.148660,0.490703,0.385535,0.402250,28.791849,135.454504,294.963130,0.708415,1.278488
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
864,COc1ccc(CCCOc2ccccc2C(=O)O)cc1Cc1cnc(N)nc1N,1.261934,2.226274,3.098315,3.947206,4.711247,5.468082,6.131069,6.587741,6.904270,...,0.635935,0.207502,0.580832,0.404492,0.436063,20.708151,113.214160,317.384829,0.453903,1.421387
865,Cc1cc2nc3c(=O)[nH]c(=O)nc-3n(C[C@H](O)[C@H](O)...,1.291948,2.270864,3.110325,3.912380,4.749721,5.738574,6.577730,7.348677,8.053773,...,0.633970,0.338187,0.486970,0.422375,0.314972,20.850575,104.977262,179.938545,0.458237,1.224317
866,Cc1nc(N)nc2c1c(CCc1ccc(C(=O)N[C@@H](CCC(=O)O)C...,1.269611,2.245309,3.111592,3.972098,4.799069,5.580713,6.453398,7.294959,7.996930,...,0.827438,0.133863,0.655519,0.381035,0.379190,39.814764,234.556438,544.913752,0.741157,1.415744
867,COc1cc(-c2nc3c(N)nc(N)nc3[nH]2)cc(OC)c1O,1.261369,2.214228,3.174200,4.054836,4.884578,5.951844,6.901664,7.653402,8.737535,...,0.709993,0.281740,0.581318,0.435876,0.190000,18.514656,71.380396,100.390236,0.564989,1.207194
